# Copy model card PDFs to `staging_sst_01.model_cards`

For each institution with a `{institution_id}_gold` schema:

1. List **registered models** in Unity Catalog under `{catalog}.{institution_id}_gold`.
2. Pick the model with the latest `creation_timestamp`.
3. Take its **latest version** (highest version number) and read the source run `run_id`.
4. Load the PDF from:

   `/Volumes/{catalog}/{institution_id}_gold/gold_volume/model_cards/<run_id>/`

   (for example, `.../model_cards/e39e2308e9da4a7aaefc71b7c5d97e61/model-card-*.pdf`).

Copies go to:

`/Volumes/staging_sst_01/model_cards/files/{institution_id}/`

Institutions with no registered model, version, or PDF are skipped. Re-running overwrites files at the same destination path.

In [ ]:
dbutils.widgets.dropdown("dry_run", "false", ["true", "false"], "Dry run (log only, no copies)")
dbutils.widgets.text("catalog", "staging_sst_01", "Unity Catalog name")
dbutils.widgets.text("dest_schema", "model_cards", "Destination schema")
dbutils.widgets.text("dest_volume", "files", "Destination volume name")

In [ ]:
import logging
import os
import sys
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Optional

from mlflow.tracking import MlflowClient
from pyspark.sql import functions as F
from pyspark.sql import types as T
from databricks.connect import DatabricksSession

# Repo src/ for edvise.shared.model_registry (Databricks Repos / bundle checkout)
for _candidate in (
    os.path.join(os.getcwd(), "src"),
    os.path.abspath(os.path.join(os.getcwd(), "..", "src")),
):
    if os.path.isdir(_candidate) and _candidate not in sys.path:
        sys.path.insert(0, _candidate)
        break

from edvise.shared.model_registry import resolve_latest_model_card_pdf

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
LOG = logging.getLogger("copy_model_cards")

spark = DatabricksSession.builder.getOrCreate()
mlflow_client = MlflowClient(registry_uri="databricks-uc")

CATALOG = dbutils.widgets.get("catalog").strip()
DEST_SCHEMA = dbutils.widgets.get("dest_schema").strip()
DEST_VOLUME = dbutils.widgets.get("dest_volume").strip()
DRY_RUN = dbutils.widgets.get("dry_run").strip().lower() in {"1", "true", "yes", "y"}

DEST_SCHEMA_FQ = f"{CATALOG}.{DEST_SCHEMA}"
DEST_VOLUME_PATH = f"/Volumes/{CATALOG}/{DEST_SCHEMA}/{DEST_VOLUME}"
INDEX_TABLE = f"{DEST_SCHEMA_FQ}.model_card_index"

LOG.info("catalog=%s dest=%s dry_run=%s", CATALOG, DEST_VOLUME_PATH, DRY_RUN)

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DEST_SCHEMA_FQ}")
spark.sql(
    f"""
    CREATE VOLUME IF NOT EXISTS {DEST_SCHEMA_FQ}.{DEST_VOLUME}
    COMMENT 'Central store for institution model card PDFs copied from gold volumes'
    """
)
LOG.info("Ensured schema and volume: %s", DEST_VOLUME_PATH)

In [ ]:
def resolve_institution_model_card(institution_id: str) -> tuple[str, str, str, str]:
    """
    UC registry -> source run id -> PDF under gold_volume/model_cards/<run_id>/.

    Returns (pdf_path, run_id, full_uc_model_name, version).
    """
    return resolve_latest_model_card_pdf(
        mlflow_client,
        catalog=CATALOG,
        institution_id=institution_id,
    )

In [ ]:
def discover_institution_ids() -> list[str]:
    """Institution IDs from schemas named {id}_gold in the catalog."""
    df = spark.sql(f"SHOW SCHEMAS IN {CATALOG} LIKE '*_gold'")
    name_col = next(
        (c for c in ("databaseName", "namespace", "schemaName") if c in df.columns),
        df.columns[0],
    )
    ids: list[str] = []
    for row in df.select(name_col).collect():
        schema = row[name_col]
        if schema.endswith("_gold"):
            ids.append(schema[: -len("_gold")])
    return sorted(set(ids))


institution_ids = discover_institution_ids()
LOG.info("Found %d institution(s) with *_gold schemas", len(institution_ids))
display(spark.createDataFrame([(i,) for i in institution_ids], ["institution_id"]))

In [ ]:
@dataclass
class CopyResult:
    institution_id: str
    model_run_id: str
    full_model_name: str
    model_version: str
    source_path: str
    dest_path: str
    status: str
    error: Optional[str] = None


results: list[CopyResult] = []
copied_at = datetime.now(timezone.utc).isoformat()

for institution_id in institution_ids:
    dest_dir = f"{DEST_VOLUME_PATH}/{institution_id}"
    try:
        src_path, model_run_id, full_model_name, model_version = (
            resolve_institution_model_card(institution_id)
        )
        LOG.info(
            "[%s] model=%s v%s run_id=%s pdf=%s",
            institution_id,
            full_model_name,
            model_version,
            model_run_id,
            src_path,
        )
    except Exception as e:
        LOG.info("[%s] skipped (no model card): %s", institution_id, e)
        results.append(
            CopyResult(
                institution_id=institution_id,
                model_run_id="",
                full_model_name="",
                model_version="",
                source_path="",
                dest_path="",
                status="skipped",
                error=str(e),
            )
        )
        continue

    dest_path = f"{dest_dir}/{os.path.basename(src_path)}"
    try:
        if not DRY_RUN:
            dbutils.fs.mkdirs(dest_dir)
        if DRY_RUN:
            LOG.info("DRY-RUN copy %s -> %s", src_path, dest_path)
            status = "dry_run"
        else:
            dbutils.fs.cp(src_path, dest_path, True)
            LOG.info("Copied %s -> %s", src_path, dest_path)
            status = "copied"
        results.append(
            CopyResult(
                institution_id,
                model_run_id,
                full_model_name,
                model_version,
                src_path,
                dest_path,
                status,
            )
        )
    except Exception as e:
        LOG.error("Failed %s -> %s: %s", src_path, dest_path, e)
        results.append(
            CopyResult(
                institution_id,
                model_run_id,
                full_model_name,
                model_version,
                src_path,
                dest_path,
                "error",
                str(e),
            )
        )

LOG.info(
    "Done. institutions=%d copies=%d skipped=%d errors=%d",
    len(institution_ids),
    sum(1 for r in results if r.status in {"copied", "dry_run"}),
    sum(1 for r in results if r.status == "skipped"),
    sum(1 for r in results if r.status == "error"),
)

In [ ]:
if results:
    summary_df = spark.createDataFrame(
        [
            (
                r.institution_id,
                r.model_run_id,
                r.full_model_name,
                r.model_version,
                r.source_path,
                r.dest_path,
                r.status,
                r.error,
                copied_at,
            )
            for r in results
        ],
        schema=T.StructType(
            [
                T.StructField("institution_id", T.StringType(), False),
                T.StructField("model_run_id", T.StringType(), False),
                T.StructField("full_model_name", T.StringType(), False),
                T.StructField("model_version", T.StringType(), False),
                T.StructField("source_path", T.StringType(), False),
                T.StructField("dest_path", T.StringType(), False),
                T.StructField("status", T.StringType(), False),
                T.StructField("error", T.StringType(), True),
                T.StructField("copied_at", T.StringType(), False),
            ]
        ),
    )
    display(summary_df.orderBy("institution_id", "dest_path"))

    if not DRY_RUN:
        (
            summary_df.withColumn("copied_at", F.to_timestamp("copied_at"))
            .write.format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(INDEX_TABLE)
        )
        LOG.info("Wrote index table %s", INDEX_TABLE)
else:
    LOG.warning("No PDFs copied; index table not updated")